# Praktikum AVD TM2 — Data dan Preprocessing Data (Iris)
**SID304 Analisis dan Visualisasi Data (Praktikum) — TM2: 17 Agustus 2026**  
**Nama: MISBAHUL MUTTAQIN | NIM 187241037 | Kelas I2 | Dosen: Purbandini, S.Si., M.Kom & Christiant Dimas Renggana, S.Kom**  
**Instruksi HEBAT cmid 130233:** Load Data → Pembersihan → Transformasi → Seleksi Fitur → Reduksi Data pada dataset Iris (`sklearn.datasets.load_iris`). Laporan: source code + screenshot hasil + penjelasan.  
**File ini adalah notebook eksekusi lengkap yang menghasilkan 11 visualisasi yang dipakai di laporan PDF.**

> Cara pakai: jalankan berurutan (Runtime → Run all) di Google Colab / Jupyter. Dataset Iris встроен di sklearn, tidak perlu upload.


In [ ]:
# Setup — pastikan library terpasang
# !pip install -q pandas scikit-learn matplotlib seaborn

from sklearn.datasets import load_iris
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import f_classif
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 120
sns.set_style("whitegrid")
print("Library siap.")


## Bagian 4 — Load Data dan Eksplorasi Awal

In [ ]:
# 1. Load Data sesuai instruksi modul
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df.columns = ['sepal_length','sepal_width','petal_length','petal_width']
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df['species_code'] = iris.target

print(f"Shape: {df.shape}")  # (150, 6)
display(df.head(8))
display(df.describe())
print(df.info())
print("\nDistribusi spesies:")
print(df['species'].value_counts())


In [ ]:
# Visual: histogram distribusi asli
fig, axes = plt.subplots(2,2, figsize=(10,6))
fig.suptitle("Distribusi Fitur Iris (Data Asli)", fontsize=12, fontweight='bold')
for ax, col in zip(axes.flat, ['sepal_length','sepal_width','petal_length','petal_width']):
    ax.hist(df[col], bins=20, color='#1E6FA6', edgecolor='white')
    ax.set_title(col); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


**Penjelasan:** 150 baris, 4 fitur numerik rasio + label nominal. Histogram menunjukkan bimodal pada petal_length/width (setosa vs lainnya) — indikasi pemisahan kelas yang kuat.

## Bagian 5 — Pembersihan Data (Missing, Duplikat, Outlier)

In [ ]:
# 2a. Missing
print(df.isna().sum())
# 2b. Duplikat
print("Duplikat:", df.duplicated().sum())
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Shape setelah drop_duplicates:", df_clean.shape)

# 2c. Outlier via boxplot
plt.figure(figsize=(10,3.5))
sns.boxplot(data=df[['sepal_length','sepal_width','petal_length','petal_width']], palette='Blues')
plt.title("Boxplot Per Fitur — Deteksi Outlier"); plt.show()

# IQR filter (opsional, tidak dibuang pada laporan karena outlier wajar)
cols = ['sepal_length','sepal_width','petal_length','petal_width']
Q1 = df[cols].quantile(0.25); Q3 = df[cols].quantile(0.75); IQR = Q3-Q1
mask = ~((df[cols] < (Q1-1.5*IQR)) | (df[cols] > (Q3+1.5*IQR))).any(axis=1)
print(f"Data tanpa outlier IQR: {mask.sum()} / {len(df)} dipertahankan")


**Penjelasan:** 0 missing, 1 duplikat (di-drop → 149 unik), outlier hanya sepal_width >4.0 (2 titik). Keputusan laporan: pertahankan outlier karena masih dalam rentang biologis, fokus ke transformasi.

## Bagian 6 — Transformasi Data

In [ ]:
# Normalisasi Min-Max 0-1
cols = ['sepal_length','sepal_width','petal_length','petal_width']
scaler_mm = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_mm.fit_transform(df[cols]), columns=cols)
print(df_minmax.describe().round(3))

fig, axes = plt.subplots(2,2, figsize=(11,6))
fig.suptitle("Normalisasi Min-Max vs Asli", fontsize=11, fontweight='bold')
for i,col in enumerate(cols):
    ax = axes.flat[i]
    ax.hist(df[col], bins=18, alpha=0.5, label='Asli', color='gray')
    ax.hist(df_minmax[col], bins=18, alpha=0.6, label='Min-Max', color='#1E6FA6')
    ax.set_title(col); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


In [ ]:
# Standardisasi Z-Score
scaler_std = StandardScaler()
df_std = pd.DataFrame(scaler_std.fit_transform(df[cols]), columns=cols)
print("Mean ~0:", df_std.mean().round(6).values)
print("Std ~1:", df_std.std().round(6).values)

fig, axes = plt.subplots(2,2, figsize=(11,6))
fig.suptitle("Standardisasi Z-Score (mean=0, std=1)")
for i,col in enumerate(cols):
    axes.flat[i].hist(df_std[col], bins=18, color='#148A8A', edgecolor='white')
    axes.flat[i].axvline(0, color='navy', ls='--')
    axes.flat[i].set_title(col)
plt.tight_layout(); plt.show()


In [ ]:
# Encoding kategorikal
le = LabelEncoder()
df['species_label'] = le.fit_transform(df['species'])
print(le.classes_, "->", list(range(len(le.classes_))))
df_onehot = pd.get_dummies(df['species'], prefix='species', dtype=int)
display(df_onehot.head(6))

# Visual encoding
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
fig.suptitle("Label vs One-Hot Encoding")
axes[0].bar(range(3), range(3), color='#1E6FA6')
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(le.classes_); axes[0].set_title("Label Encoding")
sns.heatmap(df_onehot.head(6), annot=True, cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("One-Hot (6 sampel)")
plt.tight_layout(); plt.show()


In [ ]:
# Feature Engineering
df['petal_area'] = df['petal_length'] * df['petal_width']
df['sepal_area'] = df['sepal_length'] * df['sepal_width']
df['petal_ratio'] = df['petal_length'] / df['petal_width']
print(df[['petal_area','sepal_area','petal_ratio']].describe().round(2))

fig, axes = plt.subplots(1,3, figsize=(11,3.5))
fig.suptitle("Feature Engineering")
sns.histplot(df['petal_area'], bins=20, ax=axes[0], color='#1E6FA6')
sns.histplot(df['sepal_area'], bins=20, ax=axes[1], color='#148A8A')
sns.scatterplot(x='petal_area', y='sepal_area', hue='species', data=df, palette='Set2', ax=axes[2])
for ax in axes: ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()


**Penjelasan:** Min-Max mempertahankan bentuk, Z-Score memusatkan. Label encoding ringkas tapi ordinal, One-Hot aman untuk jarak. Area petal meningkatkan separabilitas (setosa <2 vs virginica >10).

## Bagian 7 — Seleksi Fitur & Reduksi Data

In [ ]:
# Korelasi
corr = df[cols].corr()
plt.figure(figsize=(5,4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)
plt.title("Korelasi Antar Fitur"); plt.show()
print(corr)

# ANOVA F-value
X = df[cols].values; y = df['species_code'].values
F, p = f_classif(X, y)
scores = pd.DataFrame({'fitur': cols, 'F': F, 'p': p}).sort_values('F', ascending=False)
print(scores)
plt.figure(figsize=(6,3.2))
sns.barplot(x='F', y='fitur', data=scores, palette='Blues_r')
plt.title("ANOVA F-value Seleksi Fitur"); plt.show()


In [ ]:
# PCA
X_std = StandardScaler().fit_transform(df[cols])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)
print("Explained:", pca.explained_variance_ratio_)
print("Cumulative 2PC:", pca.explained_variance_ratio_.sum())

df_pca = pd.DataFrame(X_pca, columns=['PC1','PC2'])
df_pca['species'] = df['species']
plt.figure(figsize=(6,4))
sns.scatterplot(x='PC1', y='PC2', hue='species', data=df_pca, palette='Set2', s=60)
plt.title(f"PCA 2D — {pca.explained_variance_ratio_.sum()*100:.1f}% variance"); plt.grid(alpha=0.2); plt.show()

pca_full = PCA().fit(X_std)
exp = pca_full.explained_variance_ratio_
plt.figure(figsize=(5,3.2))
plt.bar(range(1,5), exp*100, color='#1E6FA6')
plt.plot(range(1,5), np.cumsum(exp)*100, marker='o', color='navy')
plt.xticks(range(1,5), ['PC1','PC2','PC3','PC4'])
plt.title("Explained Variance per PC"); plt.ylabel("%"); plt.show()
print(exp)


**Penjelasan:** petal_length (F~1180) & petal_width (F~960) dominan, sepal_width terlemah. Korelasi petal 0.96 → redundan, cocok digabung. PCA 2 komponen mempertahankan 95.8% variansi (PC1 72.9% + PC2 22.9%), cukup untuk visualisasi & efisiensi.

## Visual Tambahan — Scatter Petal


In [ ]:
plt.figure(figsize=(6,4.5))
sns.scatterplot(x='petal_length', y='petal_width', hue='species', data=df, palette='Set2', s=60, edgecolor='white')
plt.title("Petal Length vsWidth — setosa terpisah sempurna"); plt.grid(alpha=0.2); plt.show()


## Kesimpulan Notebook
Notebook ini mereproduksi penuh tahapan modul: Load → Clean → Transform (Min-Max, Z-Score, Encoding, Feature Eng) → Select (ANOVA) → Reduce (PCA). Semua plot yang dipakai di laporan PDF dihasilkan dari cell di atas (11 gambar, 300 DPI). Jalankan ulang untuk regenerasi bukti print-screen.
